# Fink/LSST — Statistics of Alerts on Heatmap

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/CNRS
- **creation date** : 2026-04-02
- **last update** : 2026-04-04
- **subject:** Fink alert Broker applied to Rubin LSST alerts

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_05"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per group
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ────────────────────────────────────────────────────────
PLOT_MODE = "all"  # 'all' | 'calib' | 'exclude'

CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "gaia_star_variable",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes)."""
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)
    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0
    log10_e_inv = 1.085736
    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)
    return lup, lup_err, b


print("flux_to_luptitude() defined.")

In [ ]:
def plot_focal_plane_heatmap_vectorized(
    alerts_df,
    geometry_csv,
    value_col="r:visit",
    agg_func="count",
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
):
    """Draw a single focal-plane heatmap (all bands combined)."""
    geom = pd.read_csv(geometry_csv)
    values = alerts_df.groupby("r:detector")[value_col].agg(agg_func)
    geom["value"] = geom["detector"].map(values).fillna(0).astype(float)

    xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()

    if log_scale:
        vmin = max(geom["value"][geom["value"] > 0].min(), 1e-2)
        vmax = geom["value"].max()
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = mcolors.Normalize(vmin=geom["value"].min(), vmax=geom["value"].max())

    cmap_obj = plt.get_cmap(cmap)
    fig, ax = plt.subplots(figsize=figsize)

    for i, row in geom.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row["value"]
        color = cmap_obj(norm(val)) if val > 0 else "lightgrey"
        poly = patches.Polygon(corners, facecolor=color, edgecolor="black", linewidth=0.2)
        ax.add_patch(poly)
        ax.text(
            row["x_center"],
            row["y_center"],
            f"{int(row['detector'])}\n{row['name']}",
            ha="center",
            va="center",
            fontsize=6,
            fontweight="bold",
        )

    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")
    ax.set_title(title)

    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

## 3. Load Parquet files from disk

In [ ]:
PATH_FOCALPLANEGEOM_filename = "data_FocalPlane/ccd_geometry.csv"
geometry_csv = PATH_FOCALPLANEGEOM_filename

In [ ]:
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))


def _is_excluded(g):
    return g in CALIBRATION_EXCLUDE or g.startswith("simbad_variable")


if PLOT_MODE == "calib":
    selected_groups = [g for g in all_groups if not _is_excluded(g)]
elif PLOT_MODE == "exclude":
    selected_groups = [g for g in all_groups if _is_excluded(g)]
else:
    selected_groups = list(all_groups)

print(f"Groups on disk: {len(all_groups)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_groups)}")
for g in all_groups:
    has_fp = "yes" if g in groups_fp else "no"
    has_src = "yes" if g in groups_src else "no"
    sel = "<<" if g in selected_groups else "  "
    label = "[EXCL] " if _is_excluded(g) else "[calib]"
    print(f"  {sel} {label} {g:40s}  fp={has_fp:3s}  src={has_src}")

In [ ]:
lc_cache = {}

for group in all_groups:
    lc_cache[group] = {}
    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

print("Objects loaded per group:")
for g in all_groups:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:35s}  {n_oids:3d} objects  {n_pts:6d} data points")

## 4. Prepare all-band alert table and heatmap

In [ ]:
def concat_group_src(lc_cache, group="gaia_star_stable"):
    """Concatenate all src DataFrames for a given group."""
    dfs = []
    for diaObjectId, content in lc_cache[group].items():
        if "src" in content and content["src"] is not None:
            df = content["src"].copy()
            df["diaObjectId"] = diaObjectId
            dfs.append(df)
    if len(dfs) == 0:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

In [ ]:
df_alerts_gaia_star_stable = concat_group_src(lc_cache, "gaia_star_stable")
df_alerts_gaia_star_stable_hq = concat_group_src(lc_cache, "gaia_star_stable_hq")
df_all_alerts_gaia_star = pd.concat(
    [df_alerts_gaia_star_stable, df_alerts_gaia_star_stable_hq], ignore_index=True
)
print(f"Total src alerts (all bands): {len(df_all_alerts_gaia_star):,}")

In [ ]:
df_all_alerts_gaia_star.groupby("r:detector")["r:visit"].count().describe()

## 5. All-band focal-plane heatmap (alert count)

In [ ]:
plot_focal_plane_heatmap_vectorized(
    df_all_alerts_gaia_star,
    geometry_csv,
    value_col="r:visit",
    agg_func="count",
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Alert counts on LSSTCam Focal Plane Heatmap (all bands)",
)

## 6. Per-band focal-plane heatmaps — 2×3 grid with shared colour scale

Alert counts are shown separately for each photometric band `u g r i z y`
(left to right, top to bottom).  A **single colour scale** is shared across all
six subplots so that differences in alert rate between bands are immediately
visible.  The colour bar is drawn once on the right of the figure.

CCD patches are labelled with their **detector number only** (no CCD name)
to avoid clutter in the smaller subplots.  X/Y axis labels are suppressed.

In [ ]:
# ── Load geometry once ────────────────────────────────────────────────────────
geom = pd.read_csv(geometry_csv)

# Compute radial distance
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# ── Focal-plane bounding box ──────────────────────────────────────────────────
fp_xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
fp_xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
fp_ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
fp_ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()

# ── Per-band alert counts ─────────────────────────────────────────────────────
band_counts = {}  # band -> Series indexed by detector
for band in BANDS:
    df_b = df_all_alerts_gaia_star[df_all_alerts_gaia_star["r:band"] == band]
    band_counts[band] = df_b.groupby("r:detector")["r:visit"].count()

# ── Global colour scale: max alert count across all bands and all CCDs ────────
global_max = max(int(s.max()) for s in band_counts.values() if len(s) > 0)
global_min = 1  # use 1 as vmin so that zero-count CCDs map to grey
norm_shared = mcolors.LogNorm(vmin=global_min, vmax=global_max)
cmap_shared = plt.get_cmap("viridis")

print(f"Global alert count range (all bands, all CCDs): [{global_min}, {global_max}]")
for band in BANDS:
    s = band_counts[band]
    if len(s) > 0:
        print(f"  band {band}: {int(s.sum()):6,} alerts  |  CCD range [{int(s.min())}, {int(s.max())}]")
    else:
        print(f"  band {band}: no data")

In [ ]:
# ── 2 rows × 3 columns subplot grid ──────────────────────────────────────────
fig_bands, axes_bands = plt.subplots(
    2,
    3,
    figsize=(18, 12),
    constrained_layout=True,
)

for idx, band in enumerate(BANDS):  # u g r i z y  → indices 0..5
    row, col = divmod(idx, 3)  # 2×3 grid, row-major
    ax = axes_bands[row, col]

    # Map alert counts onto the geometry for this band
    geom_b = geom.copy()
    geom_b["n_alerts"] = geom_b["detector"].map(band_counts[band]).fillna(0).astype(float)

    # Draw CCD polygons
    for _, row_g in geom_b.iterrows():
        corners = [(row_g[f"corner{j}_x"], row_g[f"corner{j}_y"]) for j in range(4)]
        val = row_g["n_alerts"]
        # Zero-count CCDs → light grey; use shared norm for coloured CCDs
        if val <= 0:
            face_color = "lightgrey"
        else:
            face_color = cmap_shared(norm_shared(val))
        poly = patches.Polygon(
            corners,
            facecolor=face_color,
            edgecolor="black",
            linewidth=0.2,
        )
        ax.add_patch(poly)
        # Label with detector number only (no name)
        ax.text(
            row_g["x_center"],
            row_g["y_center"],
            str(int(row_g["detector"])),
            ha="center",
            va="center",
            fontsize=5,
            fontweight="bold",
            color="k",
        )

    ax.set_aspect("equal")
    ax.set_xlim(fp_xmin, fp_xmax)
    ax.set_ylim(fp_ymin, fp_ymax)
    # Suppress x/y axis ticks and labels
    ax.set_xticks([])
    ax.set_yticks([])
    # Band label as subplot title with band colour
    n_total_band = int(band_counts[band].sum()) if len(band_counts[band]) > 0 else 0
    ax.set_title(
        rf"Band {band}   (N={n_total_band:,})",
        fontsize=11,
        color=BAND_COLORS[band],
        fontweight="bold",
    )

# ── Single shared colour bar on the right ─────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap_shared, norm=norm_shared)
sm.set_array([])
cbar = fig_bands.colorbar(sm, ax=axes_bands, fraction=0.02, pad=0.02)
cbar.set_label("Alert count per CCD  (log scale)", fontsize=11)

fig_bands.suptitle(
    "LSSTCam Focal Plane — Alert count per band\n(shared log colour scale  |  grey = no alert in this band)",
    fontsize=13,
)

savefig("focal_plane_alert_count_per_band_grid")
plt.show()

## 7. All-band radial analysis: alert count vs. $r$ — three models

Each CCD is projected onto a single radial coordinate
$r = \sqrt{x_{\rm center}^2 + y_{\rm center}^2}$ [mm].

### Three competing models

| Model | Formula | Parameters | Physical motivation |
|---|---|---|---|
| M1 — Cosine  | $N = A\cos(Br)$ | 2 | solid-angle projection, first order |
| M2 — Cosine⁴ | $N = A\cos^4(Br)$ | 2 | cos⁴ vignetting law (wide-field optics) |
| M3 — Full    | $N = N_0\cos(Kr)^\beta\left(1+r^2/r_0^2\right)^{-\gamma}$ | 5 | projection + Lorentzian vignetting |

Fits are restricted to $r \leq R_{\rm fit}$; extrapolations shown as dash-dot lines.

In [ ]:
# ── Per-CCD alert counts (all bands) ─────────────────────────────────────────
alert_counts_per_ccd = (
    df_all_alerts_gaia_star.groupby("r:detector")["r:visit"]
    .count()
    .rename("n_alerts")
    .reset_index()
    .rename(columns={"r:detector": "detector"})
)

geom_counts = geom.merge(alert_counts_per_ccd, on="detector", how="left")
geom_counts["n_alerts"] = geom_counts["n_alerts"].fillna(0).astype(float)
geom_counts["n_err"] = np.where(
    geom_counts["n_alerts"] > 0,
    np.sqrt(geom_counts["n_alerts"]),
    1.0,
)
geom_fit_all = geom_counts[geom_counts["n_alerts"] > 0].copy()

print(f"Total CCDs in geometry : {len(geom_counts)}")
print(f"CCDs with alerts       : {len(geom_fit_all)}")
print(f"r_mm range             : [{geom_fit_all['r_mm'].min():.1f},  {geom_fit_all['r_mm'].max():.1f}] mm")

In [ ]:
R_FIT_MAX = 120.0  # mm — fit range
R_FITPLOT_MAX = 150.0  # mm — extrapolation range shown in plot

# ── Model definitions ────────────────────────────────────────────────────────


def model_cos(r, A, B):
    """
    M1 — Cosine model: N(r) = A * cos(B * r)

    Parameters
    ----------
    r : array-like  — radial distance [mm]
    A : float       — on-axis normalisation
    B : float       — angular scale factor [rad/mm]
    """
    return A * np.cos(B * np.asarray(r, dtype=float))


def model_cos4(r, A, B):
    """
    M2 — Cosine-to-the-fourth model: N(r) = A * cos^4(B * r)

    The cos^4 law arises naturally in wide-field optics:
      - one cos factor from the projected solid angle of the aperture,
      - one from the obliquity of the ray bundle at the focal plane,
      - two more from the reduced lens aperture as seen from off-axis
        (Slater & Peterson 1980, Smith 2008 "Modern Optical Engineering").
    This is a stronger vignetting than the simple cos^2 Malus law and is
    commonly observed in wide-field camera systems.

    Parameters
    ----------
    r : array-like  — radial distance [mm]
    A : float       — on-axis normalisation
    B : float       — angular scale factor [rad/mm]
    """
    return A * np.cos(B * np.asarray(r, dtype=float)) ** 4


def model_full(r, N0, K, beta, r0, gamma):
    """
    M3 — Full 5-parameter model:

        N(r) = N0 * cos(K*r)^beta * (1 + (r/r0)^2)^(-gamma)

    The first factor is a generalised cosine projection with exponent beta
    (beta=1 → M1, beta=4 → M2 when K=B).
    The second factor is a Lorentzian-like vignetting / field-illumination
    envelope with scale radius r0 and power-law slope gamma.

    Parameters
    ----------
    r     : array-like
    N0    : float  — on-axis normalisation
    K     : float  — angular scale factor [rad/mm]
    beta  : float  — cosine exponent (>0)
    r0    : float  — vignetting scale radius [mm] (>0)
    gamma : float  — vignetting power-law slope (>0)
    """
    r = np.asarray(r, dtype=float)
    cos_term = np.clip(np.cos(K * r), 0.0, None)
    lorentz_term = (1.0 + (r / r0) ** 2) ** (-gamma)
    return N0 * cos_term**beta * lorentz_term


# ── Inner / outer split ───────────────────────────────────────────────────────
mask_inner = geom_fit_all["r_mm"] <= R_FIT_MAX
mask_outer = geom_fit_all["r_mm"] > R_FIT_MAX
geom_inner = geom_fit_all[mask_inner].copy()
geom_outer = geom_fit_all[mask_outer].copy()

r_data = geom_inner["r_mm"].values
N_data = geom_inner["n_alerts"].values
N_err = geom_inner["n_err"].values
N_max = N_data.max()

print(f"Fit range  : r <= {R_FIT_MAX:.0f} mm  →  {len(geom_inner)} CCDs")
print(f"Outer range: r >  {R_FIT_MAX:.0f} mm  →  {len(geom_outer)} CCDs")


def run_fit(model_func, p0, bounds, n_params, label, r_arr=None, N_arr=None, N_err_arr=None):
    """
    Weighted least-squares fit of model_func to radial alert counts.

    Uses the module-level r_data / N_data / N_err arrays by default;
    pass r_arr / N_arr / N_err_arr to override (per-band fits).

    Returns
    -------
    dict with keys: ok, popt, perr, chi2_red
    """
    r = r_data if r_arr is None else r_arr
    N = N_data if N_arr is None else N_arr
    Ne = N_err if N_err_arr is None else N_err_arr
    try:
        popt, pcov = curve_fit(
            model_func,
            r,
            N,
            p0=p0,
            sigma=Ne,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=20000,
        )
        perr = np.sqrt(np.diag(pcov))
        resid = N - model_func(r, *popt)
        chi2_red = float(np.sum((resid / Ne) ** 2) / (len(N) - n_params))
        print(f"\n{label}:")
        for pname, val, err in zip([f"p{i}" for i in range(n_params)], popt, perr):
            print(f"   {pname} = {val:.5g} +/- {err:.5g}")
        print(f"   chi2_red = {chi2_red:.3f}  (dof = {len(N) - n_params})")
        return dict(ok=True, popt=popt, perr=perr, chi2_red=chi2_red)
    except (RuntimeError, ValueError) as exc:
        print(f"WARNING: {label} fit failed: {exc}")
        return dict(ok=False, popt=None, perr=None, chi2_red=np.nan)


# ── M1 fit ────────────────────────────────────────────────────────────────────
res1 = run_fit(
    model_cos,
    p0=[N_max, 1.0 / 300.0],
    bounds=([0, 0], [np.inf, np.inf]),
    n_params=2,
    label="M1  N = A*cos(B*r)",
)

# ── M2 fit (cos^4) ────────────────────────────────────────────────────────────
res2 = run_fit(
    model_cos4,
    p0=[N_max, 1.0 / 300.0],
    bounds=([0, 0], [np.inf, np.inf]),
    n_params=2,
    label="M2  N = A*cos^4(B*r)",
)

# ── M3 fit ────────────────────────────────────────────────────────────────────
res3 = run_fit(
    model_full,
    p0=[N_max, 1.0 / 300.0, 1.0, R_FIT_MAX / 2.0, 0.5],
    bounds=([0, 0, 0, 1, 0], [np.inf, np.inf, 10, np.inf, 10]),
    n_params=5,
    label="M3  N = N0*cos(K*r)^beta * (1+(r/r0)^2)^(-gamma)",
)

In [ ]:
# ── Overlay plot: all-band data + three fit curves ────────────────────────────
M_COLORS = {"M1": "tomato", "M2": "royalblue", "M3": "forestgreen"}

r_max_plot = geom_fit_all["r_mm"].max() * 1.05
r_all = np.linspace(0, r_max_plot, 600)
r_solid = r_all[r_all <= R_FIT_MAX]
r_dash = r_all[(r_all >= R_FIT_MAX) & (r_all <= R_FITPLOT_MAX)]

fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)

vmin_c = geom_fit_all["n_alerts"].min()
vmax_c = geom_fit_all["n_alerts"].max()

sc_in = ax.scatter(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    c=geom_inner["n_alerts"],
    cmap="jet",
    vmin=vmin_c,
    vmax=vmax_c,
    s=35,
    zorder=4,
    label=rf"CCD  $r \leq {R_FIT_MAX:.0f}$ mm  (fit range)",
)
ax.errorbar(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    yerr=geom_inner["n_err"],
    fmt="none",
    ecolor="steelblue",
    elinewidth=0.8,
    capsize=2,
    alpha=0.55,
    zorder=3,
)
if len(geom_outer) > 0:
    ax.scatter(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        c=geom_outer["n_alerts"],
        cmap="jet",
        vmin=vmin_c,
        vmax=vmax_c,
        s=35,
        zorder=4,
        marker="o",
        facecolors="none",
        linewidths=1.2,
        label=rf"CCD  $r > {R_FIT_MAX:.0f}$ mm  (excluded)",
    )
    ax.errorbar(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        yerr=geom_outer["n_err"],
        fmt="none",
        ecolor="steelblue",
        elinewidth=0.8,
        capsize=2,
        alpha=0.35,
        zorder=3,
    )

if res1["ok"]:
    A1, B1 = res1["popt"]
    dA1, dB1 = res1["perr"]
    lbl1 = (
        rf"M1: $A\cos(Br)$  $\chi^2_{{\rm red}}={res1['chi2_red']:.2f}$"
        f"\n$A={A1:.0f}\\pm{dA1:.0f}$,  $B={B1:.5f}\\pm{dB1:.5f}$ rad/mm"
    )
    ax.plot(r_solid, model_cos(r_solid, A1, B1), color=M_COLORS["M1"], lw=2, ls="-", label=lbl1, zorder=5)
    ax.plot(
        r_dash,
        model_cos(r_dash, A1, B1),
        color=M_COLORS["M1"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M1 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

if res2["ok"]:
    A2, B2 = res2["popt"]
    dA2, dB2 = res2["perr"]
    lbl2 = (
        rf"M2: $A\cos^4(Br)$  $\chi^2_{{\rm red}}={res2['chi2_red']:.2f}$"
        f"\n$A={A2:.0f}\\pm{dA2:.0f}$,  $B={B2:.5f}\\pm{dB2:.5f}$ rad/mm"
    )
    ax.plot(r_solid, model_cos4(r_solid, A2, B2), color=M_COLORS["M2"], lw=2, ls="-", label=lbl2, zorder=5)
    ax.plot(
        r_dash,
        model_cos4(r_dash, A2, B2),
        color=M_COLORS["M2"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M2 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

if res3["ok"]:
    N3, K3, beta3, r03, gam3 = res3["popt"]
    dN3, dK3, db3, dr03, dgam3 = res3["perr"]
    lbl3 = (
        rf"M3: $N_0\cos(Kr)^\beta(1+r^2/r_0^2)^{{-\gamma}}$  "
        rf"$\chi^2_{{\rm red}}={res3['chi2_red']:.2f}$"
        f"\n$N_0={N3:.0f}\\pm{dN3:.0f}$,  $K={K3:.5f}\\pm{dK3:.5f}$ rad/mm"
        f"\n$\\beta={beta3:.2f}\\pm{db3:.2f}$,  "
        f"$r_0={r03:.1f}\\pm{dr03:.1f}$ mm,  $\\gamma={gam3:.2f}\\pm{dgam3:.2f}$"
    )
    ax.plot(
        r_solid, model_full(r_solid, *res3["popt"]), color=M_COLORS["M3"], lw=2, ls="-", label=lbl3, zorder=5
    )
    ax.plot(
        r_dash,
        model_full(r_dash, *res3["popt"]),
        color=M_COLORS["M3"],
        lw=1.6,
        ls="-.",
        zorder=5,
        label=rf"M3 extrap. ($r>{R_FIT_MAX:.0f}$ mm)",
    )

ax.axvline(R_FIT_MAX, color="gray", lw=1.0, ls=":", label=rf"Fit boundary $r={R_FIT_MAX:.0f}$ mm")
cbar = fig.colorbar(sc_in, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("N alerts per CCD", fontsize=8)
ax.set_xlabel(r"Radial distance from camera centre  $r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
ax.set_ylabel("Number of alerts per CCD  $N$", fontsize=10)
ax.set_title(
    rf"Alert count vs. focal-plane radius — three models, fit range $r \leq {R_FIT_MAX:.0f}$ mm",
    fontsize=11,
)
ax.set_xlim(-0.5, r_max_plot)
ax.set_ylim(bottom=0)
ax.legend(fontsize=7, loc="upper right", ncol=2)

savefig("alert_count_vs_radius_3models")
plt.show()

In [ ]:
# ── All-band model comparison table ──────────────────────────────────────────
models = [
    ("M1", "A*cos(B*r)", 2, model_cos, res1),
    ("M2", "A*cos^4(B*r)", 2, model_cos4, res2),
    ("M3", "N0*cos(K*r)^beta*(1+(r/r0)^2)^-gam", 5, model_full, res3),
]

rows = []
for mname, formula, k, func, res in models:
    if res["ok"]:
        resid = N_data - func(r_data, *res["popt"])
        chi2_tot = float(np.sum((resid / N_err) ** 2))
        rows.append(
            dict(
                model=mname,
                formula=formula,
                n_params=k,
                chi2_red=round(res["chi2_red"], 3),
                AIC=round(chi2_tot + 2 * k, 1),
            )
        )
    else:
        rows.append(dict(model=mname, formula=formula, n_params=k, chi2_red=np.nan, AIC=np.nan))

df_cmp = pd.DataFrame(rows).set_index("model")
print(f"\nModel comparison — all bands (fit range r <= {R_FIT_MAX:.0f} mm, {len(r_data)} CCDs):")
display(df_cmp)

In [ ]:
# ── Residuals — 3 models side by side ────────────────────────────────────────
active = [(mname, func, res) for mname, formula, k, func, res in models if res["ok"]]

if active:
    n_mod = len(active)
    fig_r, axes_r = plt.subplots(2, n_mod, figsize=(5 * n_mod, 8), constrained_layout=True)
    if n_mod == 1:
        axes_r = axes_r.reshape(2, 1)

    for col, (mname, func, res) in enumerate(active):
        N_pred = func(r_data, *res["popt"])
        pull = (N_data - N_pred) / N_err
        color = M_COLORS[mname]

        axes_r[0, col].scatter(r_data, pull, c=pull, cmap="RdBu_r", vmin=-3, vmax=3, s=28, zorder=3)
        axes_r[0, col].axhline(0, color="k", lw=0.8, ls="--")
        for yh in (+2, -2):
            axes_r[0, col].axhline(yh, color="gray", lw=0.6, ls=":")
        axes_r[0, col].set_xlabel(r"$r$ [mm]", fontsize=9)
        axes_r[0, col].set_ylabel(r"$(N_{\rm obs}-N_{\rm fit})/\sqrt{N_{\rm obs}}$", fontsize=8)
        axes_r[0, col].set_title(
            rf"{mname} residuals  $\chi^2_{{\rm red}}={res['chi2_red']:.2f}$", fontsize=9, color=color
        )

        axes_r[1, col].hist(pull, bins=20, color=color, alpha=0.75, edgecolor="k", linewidth=0.5)
        axes_r[1, col].axvline(0, color="k", lw=1)
        for xv in (+2, -2):
            axes_r[1, col].axvline(xv, color="gray", lw=0.8, ls=":")
        axes_r[1, col].set_xlabel(r"$(N_{\rm obs}-N_{\rm fit})/\sqrt{N_{\rm obs}}$", fontsize=8)
        axes_r[1, col].set_ylabel("Number of CCDs", fontsize=8)
        axes_r[1, col].set_title(f"mean={np.mean(pull):.2f},  std={np.std(pull):.2f}", fontsize=9)

    fig_r.suptitle(rf"Residuals — fit range $r \leq {R_FIT_MAX:.0f}$ mm  ({len(r_data)} CCDs)", fontsize=11)
    savefig("alert_count_vs_radius_residuals_3models")
    plt.show()
else:
    print("No model converged — residuals plot skipped.")

## 8. Per-band radial analysis: 2×3 grid of error plots with M3 fit

All six photometric bands `u g r i z y` are presented in a **single figure**
arranged as a 2-row × 3-column grid (left to right, top to bottom).  For each
band only **model M3** is fitted and overlaid to keep the legend concise.

- Filled markers: CCDs in the fit range $r \leq R_{\rm fit}$.
- Hollow markers: CCDs beyond $R_{\rm fit}$ (shown but excluded from the fit).
- Black solid curve: M3 fit on $[0, R_{\rm fit}]$.
- Black dash-dot curve: M3 extrapolation on $[R_{\rm fit}, R_{\rm plot}]$.

Fit parameters for all bands and the all-band case are collected in the
summary table in Section 9.

In [ ]:
# ── Per-band M3 fits ──────────────────────────────────────────────────────────
band_fit_results = {}  # band -> run_fit result dict
band_geom_data = {}  # band -> geom DataFrame (CCDs with alerts only)

for band in BANDS:
    df_b = df_all_alerts_gaia_star[df_all_alerts_gaia_star["r:band"] == band]

    if len(df_b) == 0:
        print(f"  band {band}: no alerts — skipped.")
        band_fit_results[band] = dict(ok=False, popt=None, perr=None, chi2_red=np.nan)
        continue

    counts_b = (
        df_b.groupby("r:detector")["r:visit"]
        .count()
        .rename("n_alerts")
        .reset_index()
        .rename(columns={"r:detector": "detector"})
    )
    geom_b = geom.merge(counts_b, on="detector", how="left")
    geom_b["n_alerts"] = geom_b["n_alerts"].fillna(0).astype(float)
    geom_b["n_err"] = np.where(geom_b["n_alerts"] > 0, np.sqrt(geom_b["n_alerts"]), 1.0)
    geom_b_fit = geom_b[(geom_b["n_alerts"] > 0) & (geom_b["r_mm"] <= R_FIT_MAX)].copy()

    band_geom_data[band] = geom_b[geom_b["n_alerts"] > 0].copy()

    if len(geom_b_fit) < 6:
        print(f"  band {band}: only {len(geom_b_fit)} CCDs in fit range — skipped.")
        band_fit_results[band] = dict(ok=False, popt=None, perr=None, chi2_red=np.nan)
        continue

    r_b = geom_b_fit["r_mm"].values
    N_b = geom_b_fit["n_alerts"].values
    Ne_b = geom_b_fit["n_err"].values

    res_b = run_fit(
        model_full,
        p0=[N_b.max(), 1.0 / 300.0, 1.0, R_FIT_MAX / 2.0, 0.5],
        bounds=([0, 0, 0, 1, 0], [np.inf, np.inf, 10, np.inf, 10]),
        n_params=5,
        label=f"M3  band {band}",
        r_arr=r_b,
        N_arr=N_b,
        N_err_arr=Ne_b,
    )
    band_fit_results[band] = res_b

In [ ]:
# ── Single 2×3 figure: one subplot per band ──────────────────────────────────
#
# Layout: u g r  (row 0)
#         i z y  (row 1)
#
# Each subplot shows:
#   • filled markers : CCDs in fit range  (r <= R_FIT_MAX)
#   • hollow markers : CCDs outside fit range
#   • Poisson error bars
#   • black solid curve  : M3 fit  on  [0, R_FIT_MAX]
#   • black dash-dot     : M3 extrapolation  on  [R_FIT_MAX, R_FITPLOT_MAX]
#   • grey dotted line   : fit boundary

fig_grid, axes_grid = plt.subplots(
    2,
    3,
    figsize=(18, 10),
    constrained_layout=True,
    sharey=False,  # each band has its own N scale
)

for idx, band in enumerate(BANDS):  # u g r i z y → 0..5
    row_idx, col_idx = divmod(idx, 3)  # 2×3 row-major
    ax = axes_grid[row_idx, col_idx]
    bcolor = BAND_COLORS[band]

    # ── Check data availability ───────────────────────────────────────────────
    if band not in band_geom_data:
        ax.set_visible(False)
        continue

    geom_b_all = band_geom_data[band]
    mask_in_b = geom_b_all["r_mm"] <= R_FIT_MAX
    mask_out_b = geom_b_all["r_mm"] > R_FIT_MAX
    res_b = band_fit_results[band]
    n_total_b = int(geom_b_all["n_alerts"].sum())
    r_max_b = geom_b_all["r_mm"].max() * 1.05

    # ── Inner CCDs (filled markers + error bars) ──────────────────────────────
    ax.scatter(
        geom_b_all[mask_in_b]["r_mm"],
        geom_b_all[mask_in_b]["n_alerts"],
        color=bcolor,
        s=28,
        zorder=4,
        label=rf"$r \leq {R_FIT_MAX:.0f}$ mm",
    )
    ax.errorbar(
        geom_b_all[mask_in_b]["r_mm"],
        geom_b_all[mask_in_b]["n_alerts"],
        yerr=geom_b_all[mask_in_b]["n_err"],
        fmt="none",
        ecolor=bcolor,
        elinewidth=0.7,
        capsize=1.5,
        alpha=0.55,
        zorder=3,
    )

    # ── Outer CCDs (hollow markers + error bars) ──────────────────────────────
    if mask_out_b.sum() > 0:
        ax.scatter(
            geom_b_all[mask_out_b]["r_mm"],
            geom_b_all[mask_out_b]["n_alerts"],
            facecolors="none",
            edgecolors=bcolor,
            s=28,
            linewidths=1.0,
            zorder=4,
            label=rf"$r > {R_FIT_MAX:.0f}$ mm",
        )
        ax.errorbar(
            geom_b_all[mask_out_b]["r_mm"],
            geom_b_all[mask_out_b]["n_alerts"],
            yerr=geom_b_all[mask_out_b]["n_err"],
            fmt="none",
            ecolor=bcolor,
            elinewidth=0.7,
            capsize=1.5,
            alpha=0.35,
            zorder=3,
        )

    # ── M3 fit and extrapolation ──────────────────────────────────────────────
    if res_b["ok"]:
        N0b, Kb, betab, r0b, gamb = res_b["popt"]
        dN0b, dKb, dbetab, dr0b, dgamb = res_b["perr"]
        # Compact two-line legend label
        lbl_fit = (
            rf"M3  $\chi^2_{{\rm red}}={res_b['chi2_red']:.2f}$"
            f"\n$N_0={N0b:.0f}$, $K={Kb:.4f}$, $\\beta={betab:.2f}$"
            f"\n$r_0={r0b:.1f}$ mm, $\\gamma={gamb:.2f}$"
        )
        r_plot = np.linspace(0, r_max_b, 500)
        r_s = r_plot[r_plot <= R_FIT_MAX]
        r_d = r_plot[(r_plot > R_FIT_MAX) & (r_plot <= R_FITPLOT_MAX)]
        ax.plot(r_s, model_full(r_s, *res_b["popt"]), color="k", lw=1.8, ls="-", label=lbl_fit, zorder=5)
        if len(r_d) > 0:
            ax.plot(r_d, model_full(r_d, *res_b["popt"]), color="k", lw=1.4, ls="-.", zorder=5)

    # ── Fit boundary ──────────────────────────────────────────────────────────
    ax.axvline(R_FIT_MAX, color="gray", lw=0.8, ls=":")

    # ── Axes labels and title ─────────────────────────────────────────────────
    ax.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=9)
    ax.set_ylabel("N alerts / CCD", fontsize=9)
    ax.set_title(
        rf"Band {band}   (N={n_total_b:,})",
        fontsize=11,
        color=bcolor,
        fontweight="bold",
    )
    ax.set_xlim(-0.5, r_max_b)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=6.5, loc="upper right")

fig_grid.suptitle(
    rf"Alert count vs. focal-plane radius — M3 fit per band  "
    rf"(fit range $r \leq {R_FIT_MAX:.0f}$ mm)",
    fontsize=13,
)

savefig("alert_count_vs_radius_M3_per_band_grid")
plt.show()

## 9. Summary table: M3 fit parameters per band

In [ ]:
# ── Summary table: M3 fit parameters for each band + all-band ────────────────
PARAM_NAMES = ["N0", "K [rad/mm]", "beta", "r0 [mm]", "gamma"]
ERR_NAMES = ["dN0", "dK", "dbeta", "dr0", "dgamma"]

summary_rows = []
for band in BANDS:
    res_b = band_fit_results.get(band, {"ok": False})
    n_total_b = int(band_geom_data[band]["n_alerts"].sum()) if band in band_geom_data else 0
    n_ccd_b = int((band_geom_data[band]["n_alerts"] > 0).sum()) if band in band_geom_data else 0

    if res_b["ok"]:
        row = {"band": band, "N_alerts": n_total_b, "N_CCD": n_ccd_b, "chi2_red": round(res_b["chi2_red"], 3)}
        for pname, ename, val, err in zip(PARAM_NAMES, ERR_NAMES, res_b["popt"], res_b["perr"]):
            row[pname] = round(float(val), 5)
            row[ename] = round(float(err), 5)
    else:
        row = {"band": band, "N_alerts": n_total_b, "N_CCD": n_ccd_b, "chi2_red": np.nan}
        for pname, ename in zip(PARAM_NAMES, ERR_NAMES):
            row[pname] = np.nan
            row[ename] = np.nan
    summary_rows.append(row)

# All-band M3 result
if res3["ok"]:
    row_all = {
        "band": "all",
        "N_alerts": int(df_all_alerts_gaia_star["r:visit"].count()),
        "N_CCD": int((geom_counts["n_alerts"] > 0).sum()),
        "chi2_red": round(res3["chi2_red"], 3),
    }
    for pname, ename, val, err in zip(PARAM_NAMES, ERR_NAMES, res3["popt"], res3["perr"]):
        row_all[pname] = round(float(val), 5)
        row_all[ename] = round(float(err), 5)
else:
    row_all = {
        "band": "all",
        "N_alerts": 0,
        "N_CCD": 0,
        "chi2_red": np.nan,
        **{p: np.nan for p in PARAM_NAMES + ERR_NAMES},
    }
summary_rows.append(row_all)

df_band_fits = pd.DataFrame(summary_rows).set_index("band")
print("\nM3 fit results per band (and all-band combined):")
display(df_band_fits)

csv_path = os.path.join(DIR_FIGS, "M3_fit_results_per_band.csv")
df_band_fits.to_csv(csv_path)
print(f"\nSaved to: {csv_path}")